# Paper hybrid results: Table 7 + Figures 6 and 7

1. **Table 7**: each generator added to the full TPC-DS log (`N = 1,012`), relative improvement and
   log-only → hybrid values on six metrics, averaged across models.
2. **Figure 6**: Log MAE reduction from adding each generator, at every log size (`N` = 5 to 1,012).
3. **Figure 7**: Log MAE reduction from adding DiverSQL, per model, across log sizes.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists() and (p / "qpp").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from qpp.hybrid import sweep

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

RESULTS_DIR = Path(sweep.__file__).resolve().parent / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df_raw, config, STAMP = sweep.load_results(RESULTS_DIR)
METRIC = "log_mae"
BASE = "D0_real_only"

SEED_AGG = "mean"
assert SEED_AGG in ("mean", "median")

MODELS = sorted(df_raw.model.unique())
DISPLAY_NAMES = {
    "gp_rbf": "GP-RBF", "lightgbm": "LightGBM", "mlp": "MLP",
    "mscn": "MSCN", "random_forest": "Random Forest", "xgboost": "XGBoost",
}
GENERATOR_ARMS = {
    "D1_hybrid_diversql": "DiverSQL", "D2_hybrid_sqlbarber": "SQLBarber",
    "D3_hybrid_bootstrapping_lcm": "Bootstrapping-LCM", "D4_hybrid_sql_factory": "SQL-Factory",
    "D5_hybrid_e2etune": "E2ETune", "D6_hybrid_sqlstorm": "SQLStorm",
}
# Alphabetical with DiverSQL last, as in the paper's other results tables.
GENERATOR_ORDER = sorted(n for n in GENERATOR_ARMS.values() if n != "DiverSQL") + ["DiverSQL"]
MODEL_COLOR = {
    "gp_rbf": "#377eb8", "lightgbm": "#4daf4a", "mlp": "#984ea3",
    "mscn": "#e41a1c", "random_forest": "#a65628", "xgboost": "#f781bf",
}

def paired(frame, arm, base=BASE, metric=METRIC):
    """Pair `arm` with `base` by (model, n_observed, seed) on their common held-out legs."""
    c = sweep.common_leg_metric(frame, [base, arm], metric=metric)
    p = c.pivot_table(index=["model", "n_observed", "reveal_seed"], columns="arm",
                      values=metric).dropna()
    p["pct_improve"] = (p[base] - p[arm]) / p[base] * 100
    return p

print(f"run {STAMP}: {config['n_seeds']} seeds x {len(MODELS)} models, seeds aggregated by {SEED_AGG}")

## 1. Table 7: generator comparison across metrics at N = 1,012


In [ ]:
N_TABLE = 1012
frame = df_raw[df_raw.n_observed == N_TABLE]

SUMMARY_METRICS = [
    ("log_mae", r"Log\\MAE", False, 3),
    ("log_rmse", r"Log\\RMSE", False, 3),
    ("median_q_error", r"Median\\Q-Error", False, 2),
    ("gmean_q_error", r"Mean\\Q-Error", False, 2),
    ("p90_q_error", r"P90\\Q-Error", False, 2),
    ("spearman", "Spearman", True, 3),
]
SUMMARY_ORDER = GENERATOR_ORDER

# Same aggregation as the cross-generator figure: improvement per (model, seed), mean over seeds,
# then mean over models.
TABLE_SEED_AGG = "mean"

summary = {}  # (generator, metric) -> (improvement, models improved, real-only, hybrid)
per_model_abs = {}  # (generator, metric) -> per-model real-only / hybrid values
for arm, name in GENERATOR_ARMS.items():
    for metric, _, higher, _ in SUMMARY_METRICS:
        p = paired(frame, arm, metric=metric)
        m = p.groupby(level="model")[[BASE, arm]].agg(TABLE_SEED_AGG)
        d0, d1 = m[BASE].mean(), m[arm].mean()
        if higher:
            # Spearman: % increase from the averaged values -- per-seed ratios blow up when real-only
            # Spearman is near 0 (as low as 0.06), so they can't be averaged like the error metrics.
            per_model = (m[arm] - m[BASE]) / m[BASE].abs() * 100
            gain = (d1 - d0) / abs(d0) * 100
        else:
            per_model = ((p[BASE] - p[arm]) / p[BASE] * 100).groupby(level="model").agg(TABLE_SEED_AGG)
            gain = per_model.mean()
        summary[name, metric] = (gain, int((per_model > 0).sum()), d0, d1)
        per_model_abs[name, metric] = m.rename(columns={BASE: "real_only", arm: "hybrid"})

display(pd.DataFrame({m: {n: f"{summary[n, m][0]:.2f} ({summary[n, m][1]}/{len(MODELS)})"
                          for n in SUMMARY_ORDER} for m, *_ in SUMMARY_METRICS}))

In [ ]:
def _bold_if(s, cond):
    return r"\underline{\textbf{" + s + "}}" if cond else s

def _fmt_gain(v, higher):
    s = f"{abs(v):.1f}"
    return ("$-$" if v < 0 and float(s) > 0 else "") + s

# Layout for one column of the ACM two-column format, scaled to fit as in
# cold/02_paper_qpp_table_and_plot.ipynb. The absolute (bottom) line sets the width, so it
# is rounded (ABS_DECIMALS), the column padding is narrowed (TABCOLSEP) and the "(k/6)" counts are off.
TABLE_ENV = "table"            # "table" = one column; "table*" = spans both columns
TABLE_WIDTH = r"\linewidth"    # scales the tabular to this width; None = natural size
TABLE_FONT = None              # e.g. r"\small" to shrink the whole table without scaling
ARRAY_STRETCH = "0.85"
TABCOLSEP = "3pt"              # padding either side of each column (LaTeX default 6pt); None = default
SHOW_COUNTS = False            # "(k/6)" models-improved count after each improvement
COUNT_SIZE = r"\footnotesize"  # size of the "(k/6)" count
SUBTEXT_SIZE = r"\footnotesize"  # size of the real-only -> hybrid line
LABEL_SIZE = r"\footnotesize"    # size of the hybrid-partner names (row labels); None = normal
ABS_DECIMALS = {"log_mae": 2, "log_rmse": 2, "median_q_error": 1, "gmean_q_error": 1,
                "p90_q_error": 1, "spearman": 2}  # decimals on the absolute line
WRAP_LABEL_AT = None           # row labels longer than this split over two lines (None = never)
ROW_GAP_BASELINES = "2pt"      # extra space between baseline generator rows (booktabs default 0.5em)
ROW_GAP_OURS = None            # extra space before the DiverSQL row; None = none
METRICS_ROW_GAP = "-1.5pt"     # gap between each row's % line and its absolute line (less negative = more space)
LABEL = "tab:hybrid"
PLACEMENT = "h!"
VSPACE = "-1em"                # \vspace after the caption and before \end{table}; None = none
N_TEX = f"{N_TABLE:,}".replace(",", "{,}")
CAPTION = (r"Improvement from adding each generated workload to the historical log, averaged across models. "
           r"Top: relative improvement (\%). Bottom: log-only $\rightarrow$ hybrid.")

best = {m: max(round(summary[n, m][0], 1) for n in SUMMARY_ORDER) for m, *_ in SUMMARY_METRICS}

header = [r"\makecell[c]{Hybrid\\Workload}"] + [rf"\makecell[c]{{{h}}}" for _, h, _, _ in SUMMARY_METRICS]
lines = [rf"\begin{{{TABLE_ENV}}}[{PLACEMENT}]", rf"\caption{{{CAPTION}}}"]
if VSPACE:
    lines.append(rf"\vspace{{{VSPACE}}}")
lines += [rf"\label{{{LABEL}}}", r"\centering"]
if TABLE_FONT:
    lines.append(TABLE_FONT)
if TABLE_WIDTH:
    lines.append(rf"\resizebox{{{TABLE_WIDTH}}}{{!}}{{%")
if TABCOLSEP:
    lines.append(rf"\setlength{{\tabcolsep}}{{{TABCOLSEP}}}")
lines += [rf"\renewcommand{{\arraystretch}}{{{ARRAY_STRETCH}}}",
         r"\begin{tabular}{l" + "c" * len(SUMMARY_METRICS) + "}", r"\toprule",
         " & ".join(header) + r" \\", r"\midrule"]
for i, name in enumerate(SUMMARY_ORDER):
    top, bot = [], []
    for metric, _, higher, dec in SUMMARY_METRICS:
        gain, n_up, d0, d1 = summary[name, metric]
        is_best = round(gain, 1) == best[metric]
        count = " {" + COUNT_SIZE + "(" + f"{n_up}/{len(MODELS)}" + ")}" if SHOW_COUNTS else ""
        top.append(_bold_if(_fmt_gain(gain, higher), is_best) + count)
        dec = ABS_DECIMALS.get(metric, dec)
        bot.append(SUBTEXT_SIZE + " " + f"{d0:.{dec}f}" + r"$\to$" + f"{d1:.{dec}f}")
    label = name + " (Ours)" if name == "DiverSQL" else name
    if WRAP_LABEL_AT and len(label) > WRAP_LABEL_AT and " " in label:
        label = r"\makecell[l]{" + label.replace(" ", r"\\", 1) + "}"
    if LABEL_SIZE:
        label = "{" + LABEL_SIZE + " " + label + "}"
    lines.append(r"\multirow{2}{*}{" + label + "} & " + " & ".join(top) + r" \\[" + METRICS_ROW_GAP + "]")
    lines.append(" & " + " & ".join(bot) + r" \\")
    if i < len(SUMMARY_ORDER) - 2 and ROW_GAP_BASELINES:
        lines.append(rf"\addlinespace[{ROW_GAP_BASELINES}]")
    elif i == len(SUMMARY_ORDER) - 2 and ROW_GAP_OURS:
        lines.append(rf"\addlinespace[{ROW_GAP_OURS}]")
lines += [r"\bottomrule", r"\end{tabular}"] + (["}"] if TABLE_WIDTH else [])
if VSPACE:
    lines.append(rf"\vspace{{{VSPACE}}}")
lines.append(rf"\end{{{TABLE_ENV}}}")
metrics_table = "\n".join(lines)
print(metrics_table)
(FIGURES_DIR / "table7_hybrid_generator_metrics.tex").write_text(metrics_table + "\n")


## 2. Figure 6: Log MAE reduction from each generator across log sizes

Each generator's hybrid arm (`D1`–`D6`) against the log alone at every log size, scored on the legs
neither arm trained on. Bars show the mean over models of the per-model mean over seeds; colour is the
generator and shade the log size (lightest = 5 queries, darkest = the full 1,012).


In [ ]:
SCALES = sorted(n for n in df_raw.n_observed.unique() if n > 0)
cross = {}
for n in SCALES:
    frame = df_raw[df_raw.n_observed == n]
    cols = {}
    for arm, name in GENERATOR_ARMS.items():
        cols[name] = paired(frame, arm).groupby(level="model")["pct_improve"].mean()
    t = pd.DataFrame(cols)
    t.index = [DISPLAY_NAMES[m] for m in t.index]
    t.loc["Mean"] = t.mean()
    cross[n] = t

font_size = 15

# mean over models, one row per generator, one column per scale
cross_scale = pd.DataFrame({n: cross[n].loc["Mean"] for n in SCALES})

# per-generator colours, same as the cold-start figures (results_lib.LEG_COLORS)
GEN_COLOR = {"DiverSQL": "#ea580c", "SQLBarber": "#a855f7", "Bootstrapping-LCM": "#6366f1",
             "SQL-Factory": "#0891b2", "E2ETune": "#059669", "SQLStorm": "#db2777"}

names = GENERATOR_ORDER
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(names))
w = 0.8 / len(SCALES)
# bar colour = generator, shade = real-data scale (light = few real queries, full = N=1012)
shades = np.linspace(0.25, 1.0, len(SCALES))
for i, (n, shade) in enumerate(zip(SCALES, shades)):
    ax.bar(x + (i - (len(SCALES) - 1) / 2) * w, cross_scale.loc[names, n].values, w * 0.92,
           color=[GEN_COLOR[nm] for nm in names], alpha=shade, edgecolor="white", linewidth=0.3)
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x, [nm.replace("Bootstrapping-", "Bootstrapping-\n") for nm in names])
ax.set_xlim(-0.6, len(names) - 0.4)
ax.set_ylim(top=33)  # headroom so the legend clears the bars
ax.yaxis.set_major_locator(plt.MultipleLocator(5))
ax.set_ylabel("Log MAE reduction vs. log-only (%)", fontsize=font_size)
ax.tick_params(axis="x", labelsize=font_size, pad=2)
ax.tick_params(axis="y", labelsize=15.5, pad=2)
for spine in ax.spines.values():  # black frame on all four sides, as in Figure 7
    spine.set_visible(True)
    spine.set_color("black")

from matplotlib.patches import Patch
# Neutral swatches: bar colours encode the generator, shade encodes N. A near-black base at the bars'
# opacities spans about the same light-to-dark range as the coloured bars, so the steps stay distinct.
ax.legend(handles=[Patch(color="0.1", alpha=a, label=f"{1000 if n == 1012 else n:,}")  # full log (1,012) shown rounded
                   for n, a in zip(SCALES, shades)],
          title="Real Training Queries", fontsize=14, title_fontsize=font_size, ncol=len(SCALES), loc="upper right",
          columnspacing=0.8, labelspacing=0.2, handlelength=1.2, handletextpad=0.4, borderpad=0.4,
          framealpha=0.9, edgecolor="0.8")
ax.grid(False, axis="x")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure6_hybrid_generator_scaling.png", dpi=150, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "figure6_hybrid_generator_scaling.pdf", bbox_inches="tight")
plt.show()

## 3. Figure 7: DiverSQL error reduction per model across log sizes

One line per model: the paired % reduction in Log MAE from adding DiverSQL (`D1` vs. `D0`), aggregated over
seeds by `SEED_AGG`. The dashed line is a log-linear fit over all models and log sizes.


In [ ]:
import matplotlib as mpl

piv = paired(df_raw, "D1_hybrid_diversql")
pct = piv.groupby(["model", "n_observed"])["pct_improve"].agg(SEED_AGG).reset_index()

slope, intercept = np.polyfit(np.log10(pct.n_observed), pct.pct_improve, 1)

def plot_across_scale(
    pct,
    *,
    figsize=(8.2, 3.5),
    ylim=None,                  # (bottom, top); raise top to make headroom so the legend clears the data
    xlim=None,
    label_fontsize=13,
    tick_fontsize=15.5,         # matches analysis/workload_runtime_distributions.ipynb
    tick_pad=2,
    line_width=2,
    marker_size=4,
    show_trend=True,
    trend_in_legend=True,       # False = draw the trend line but leave it out of the legend
    legend_loc="upper right",
    legend_anchor=None,         # (x, y) in axes coords, e.g. (1.0, 1.0); None = matplotlib default for loc
    legend_fontsize=12,
    legend_ncol=2,
    legend_columnspacing=0.8,
    legend_labelspacing=0.1,
    legend_handlelength=1.5,
    legend_handletextpad=0.4,
    legend_borderpad=0.3,
    legend_borderaxespad=0.5,
    legend_framealpha=0.9,
    legend_edgecolor="0.8",
    legend_markers=False,       # show the point markers on legend lines
    trend_own_column=False,     # trend alone in an extra column to the right of the model grid
    trend_row="middle",         # with trend_own_column: "top", "middle", "bottom", or a row index
    trend_centred=True,         # trend entry centred under the model columns (False = last grid cell)
    trend_gap=1,              # vertical gap (points) between the model entries and the trend entry
    trend_pad_after=1.5,        # extra space (points) below the trend entry, inside the border
    trend_text_centred=True,    # centre the "Trend" text itself, with the dash hanging to its left
    trend_offset=40,          # manual horizontal shift (points, + = left) overriding trend_text_centred
    full_border=True,           # black frame on all four sides, as in cold/02_paper_qpp_table_and_plot.ipynb
    save=True,
):
    """Per-model % log-MAE reduction from adding DiverSQL vs. real-data scale, with a log-linear trend."""
    from matplotlib.offsetbox import DrawingArea, HPacker, VPacker

    fig, ax = plt.subplots(figsize=figsize)
    for model in MODELS:
        s = pct[pct.model == model].sort_values("n_observed")
        ax.plot(s.n_observed, s.pct_improve, marker="o", ms=marker_size, lw=line_width,
                color=MODEL_COLOR[model], label=DISPLAY_NAMES[model])
    if show_trend:
        slope, intercept = np.polyfit(np.log10(pct.n_observed), pct.pct_improve, 1)
        x_fit = np.logspace(np.log10(pct.n_observed.min()), np.log10(pct.n_observed.max()), 100)
        ax.plot(x_fit, intercept + slope * np.log10(x_fit), color="black", lw=1.5, ls="--", alpha=0.65,
                label="Trend" if trend_in_legend else "_nolegend_")

    ax.set_xscale("log")
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xlabel("Real Training Queries", fontsize=label_fontsize)
    ax.set_ylabel("Log MAE reduction vs. log-only (%)",
                  fontsize=label_fontsize-1)
    ax.tick_params(axis="both", labelsize=tick_fontsize, pad=tick_pad)
    if full_border:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")

    handles, labels = ax.get_legend_handles_labels()
    legend_trend = show_trend and trend_in_legend
    own_column = legend_trend and trend_own_column
    split = legend_trend and trend_centred and not own_column
    if own_column:
        # Pad the trend's column with invisible entries so the models fill whole columns and the
        # trend sits alone in the last one, at row `trend_row`.
        n_rows = int(np.ceil((len(handles) - 1) / legend_ncol))
        row = {"top": 0, "middle": (n_rows - 1) // 2, "bottom": n_rows - 1}.get(trend_row, trend_row)
        blank = mpl.patches.Rectangle((0, 0), 1, 1, visible=False)
        handles = handles[:-1] + [blank] * row + handles[-1:] + [blank] * (n_rows - 1 - row)
        labels = labels[:-1] + [""] * row + labels[-1:] + [""] * (n_rows - 1 - row)
    kw = dict(fontsize=legend_fontsize, columnspacing=legend_columnspacing,
              labelspacing=legend_labelspacing, handlelength=legend_handlelength,
              handletextpad=legend_handletextpad, borderpad=legend_borderpad,
              borderaxespad=legend_borderaxespad, framealpha=legend_framealpha,
              edgecolor=legend_edgecolor, loc=legend_loc)
    if legend_anchor is not None:
        kw["bbox_to_anchor"] = legend_anchor
    leg = ax.legend(handles[:-1] if split else handles, labels[:-1] if split else labels,
                    ncol=legend_ncol + own_column, **kw)
    if split:
        # Stack a one-entry trend legend under the model grid inside the same bordered box.
        trend = mpl.legend.Legend(ax, handles[-1:], labels[-1:], **kw)
        # Shift left by the dash + its gap so the text, not the whole entry, sits on the centre line.
        shift = trend_offset if trend_offset is not None else (
            (legend_handlelength + legend_handletextpad) * legend_fontsize if trend_text_centred else 0.0)
        spacer = DrawingArea(abs(shift), 0)
        trend_box = trend._legend_box._children[-1]
        trend_row = HPacker(children=[trend_box, spacer] if shift >= 0 else [spacer, trend_box],
                            align="center", pad=0, sep=0)
        trend_block = VPacker(children=[trend_row, DrawingArea(0, trend_pad_after)],
                              align="center", pad=0, sep=0)
        leg._legend_box = VPacker(children=[leg._legend_box, trend_block],
                                  align="center", pad=0, sep=trend_gap)
        leg._legend_box.set_figure(fig)
        leg._legend_box.set_offset(leg._findoffset)  # keep loc / bbox_to_anchor positioning
        leg.legend_handles.extend(trend.legend_handles)
    if not legend_markers:
        for h in leg.legend_handles:
            if hasattr(h, "set_marker"):
                h.set_marker("None")

    plt.tight_layout()
    if save:
        plt.savefig(FIGURES_DIR / "figure7_hybrid_diversql_by_model.png", dpi=150,
                    bbox_inches="tight")
        plt.savefig(FIGURES_DIR / "figure7_hybrid_diversql_by_model.pdf", bbox_inches="tight")
    plt.show()
    return fig, ax


plot_across_scale(pct, ylim=(None, 32), trend_own_column=True, trend_in_legend=False)

by_scale = pct.pivot(index="model", columns="n_observed", values="pct_improve")
by_scale.index = [DISPLAY_NAMES[m] for m in by_scale.index]
by_scale["Mean"] = by_scale.mean(axis=1)
by_scale.loc["Mean"] = by_scale.mean()
print(f"Trend: {slope:+.1f} pts per 10x more real data. "
      f"Mean reduction over all models and scales: {by_scale.loc['Mean', 'Mean']:.1f}%")
